In [4]:
import csv

def multiplier_oracle(a1, a0, b1, b0):
    """
    Oracle function: computes the expected multiplication result directly in Python.
    """
    # Convert binary bits to decimal integers
    A = (a1 << 1) | a0
    B = (b1 << 1) | b0

    # Perform mathematical multiplication
    P = A * B

    # Extract the decimal result back to 4-bit binary (P3, P2, P1, P0)
    p0 = P & 1
    p1 = (P >> 1) & 1
    p2 = (P >> 2) & 1
    p3 = (P >> 3) & 1

    return p3, p2, p1, p0

def validate_truth_table(csv_filename):
    print(f"Starting truth table validation: {csv_filename} ...\n")

    try:
        with open(csv_filename, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)

            # Clean up potential spaces in column names
            fieldnames = [name.strip() for name in reader.fieldnames]
            reader.fieldnames = fieldnames

            row_count = 0

            for row in reader:
                row_count += 1

                # Read current input vector
                a0 = int(row['A0'].strip())
                a1 = int(row['A1'].strip())
                b0 = int(row['B0'].strip())
                b1 = int(row['B1'].strip())

                # Read actual outputs generated by BENCH simulation
                actual_p0 = int(row['P0'].strip())
                actual_p1 = int(row['P1'].strip())
                actual_p2 = int(row['P2'].strip())
                actual_p3 = int(row['P3'].strip())

                # Compute expected outputs using the Oracle
                exp_p3, exp_p2, exp_p1, exp_p0 = multiplier_oracle(a1, a0, b1, b0)

                # Compare actual outputs with expected outputs
                if (actual_p0 != exp_p0) or (actual_p1 != exp_p1) or \
                   (actual_p2 != exp_p2) or (actual_p3 != exp_p3):

                    print("-" * 60)
                    print("Mismatch found!")
                    print(f"First failing input vector occurred at row {row_count}:")
                    print(f"Input vector (A1 A0, B1 B0): {a1}{a0}, {b1}{b0} (Decimal: {a1<<1 | a0} * {b1<<1 | b0})")
                    print(f"Expected output (Oracle P3 P2 P1 P0) : {exp_p3} {exp_p2} {exp_p1} {exp_p0}")
                    print(f"Actual output   (BENCH  P3 P2 P1 P0) : {actual_p3} {actual_p2} {actual_p1} {actual_p0}")
                    print("-" * 60)
                    print("\nLikely cause analysis:")
                    print("1. CNF Logic Synthesis Error: The generated Verilog logic (or upstream CNF equations) does not correctly reflect the boolean logic of a 2-bit multiplier.")
                    print("2. I/O Polarity or Minterm Issue: For example, an all-zero input (0 * 0) erroneously produces P1=1 and P2=1. This strongly indicates that certain minterms or the polarity of NOT gates were incorrectly mapped during CNF-to-Verilog conversion.")

                    return # Stop at the first failure as required

            print("✅ Validation passed! All rows match the oracle.")

    except FileNotFoundError:
        print(f"Error: File '{csv_filename}' not found. Please check the path and filename.")
    except KeyError as e:
        print(f"Error: Missing column in CSV file - {e}. Please check your CSV headers.")

# Run the validation
validate_truth_table('multiplier_2-bit_typ_tab.csv')

Starting truth table validation: multiplier_2-bit_typ_tab.csv ...

------------------------------------------------------------
Mismatch found!
First failing input vector occurred at row 1:
Input vector (A1 A0, B1 B0): 00, 00 (Decimal: 0 * 0)
Expected output (Oracle P3 P2 P1 P0) : 0 0 0 0
Actual output   (BENCH  P3 P2 P1 P0) : 0 1 1 0
------------------------------------------------------------

Likely cause analysis:
1. CNF Logic Synthesis Error: The generated Verilog logic (or upstream CNF equations) does not correctly reflect the boolean logic of a 2-bit multiplier.
2. I/O Polarity or Minterm Issue: For example, an all-zero input (0 * 0) erroneously produces P1=1 and P2=1. This strongly indicates that certain minterms or the polarity of NOT gates were incorrectly mapped during CNF-to-Verilog conversion.
